# 使用Subclass API构建动态模型（PyTorch版）

> 本笔记本是 [TensorFlow/Keras版本](./04-子类API构建动态模型.ipynb) 的PyTorch等价实现。
> 核心映射：`keras.Model` → `nn.Module`，`call()` → `forward()`，`Dense` → `Linear`，`Concatenate` → `torch.cat`

本教程介绍PyTorch的Module子类化方式，这是最灵活但也最需要手动控制的模型构建方法。

## 学习目标

1. 理解`nn.Module`子类化的基本语法
2. 掌握自定义模型的构建方法
3. 学会在`forward`方法中实现动态计算逻辑
4. 了解PyTorch与Keras子类化API的异同
5. 编写标准训练循环并支持GPU加速

## 三种构建方式对比

| 构建方式 | 灵活性 | 调试难度 | 适用场景 |
|---------|--------|----------|----------|
| nn.Sequential | 低 | 简单 | 线性堆叠模型 |
| nn.ModuleList/ModuleDict | 中 | 中等 | 静态多输入/输出 |
| nn.Module子类化 | 高 | 复杂 | 动态计算图、研究 |

## 1. 环境配置与数据准备

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# 设置随机种子 / Set random seeds
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# 设备配置 / Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch版本 / PyTorch version: {torch.__version__}")
print(f"计算设备 / Device: {device}")

In [ ]:
# 加载并预处理数据 / Load and preprocess data
housing = fetch_california_housing()

X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=RANDOM_SEED
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=RANDOM_SEED
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

# 准备多输入数据 / Prepare multi-input data
X_train_A, X_train_B = X_train[:, :5], X_train[:, 2:]
X_valid_A, X_valid_B = X_valid[:, :5], X_valid[:, 2:]
X_test_A, X_test_B = X_test[:, :5], X_test[:, 2:]

print(f"训练集 / Training set: {X_train.shape}")
print(f"Wide输入 / Wide input: {X_train_A.shape}, Deep输入 / Deep input: {X_train_B.shape}")

# 转换为PyTorch张量 / Convert to PyTorch tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
X_valid_t = torch.FloatTensor(X_valid)
y_valid_t = torch.FloatTensor(y_valid).unsqueeze(1)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1)

X_train_A_t = torch.FloatTensor(X_train_A)
X_train_B_t = torch.FloatTensor(X_train_B)
X_valid_A_t = torch.FloatTensor(X_valid_A)
X_valid_B_t = torch.FloatTensor(X_valid_B)
X_test_A_t = torch.FloatTensor(X_test_A)
X_test_B_t = torch.FloatTensor(X_test_B)

# 创建DataLoader / Create DataLoaders
BATCH_SIZE = 32

train_dataset = TensorDataset(X_train_t, y_train_t)
valid_dataset = TensorDataset(X_valid_t, y_valid_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 多输入DataLoader / Multi-input DataLoader
train_wd_dataset = TensorDataset(X_train_A_t, X_train_B_t, y_train_t, y_train_t)
valid_wd_dataset = TensorDataset(X_valid_A_t, X_valid_B_t, y_valid_t, y_valid_t)
test_wd_dataset = TensorDataset(X_test_A_t, X_test_B_t, y_test_t, y_test_t)

train_wd_loader = DataLoader(train_wd_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_wd_loader = DataLoader(valid_wd_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_wd_loader = DataLoader(test_wd_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 2. nn.Module子类化基础

### 核心概念

PyTorch的Module子类化需要：
1. 继承`nn.Module`类
2. 在`__init__`中定义层（调用`super().__init__()`）
3. 在`forward`方法中实现前向传播逻辑

### 基本模板

```python
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        # 在这里定义层 / Define layers here
        self.dense1 = nn.Linear(in_features, 30)
        self.output_layer = nn.Linear(30, 1)
    
    def forward(self, x):
        # 在这里定义前向传播逻辑 / Define forward pass here
        x = torch.relu(self.dense1(x))
        return self.output_layer(x)
```

### 与Keras的关键区别

| Keras | PyTorch | 说明 |
|-------|---------|------|
| `keras.Model` | `nn.Module` | 基类不同 |
| `call(self, inputs, training=None)` | `forward(self, x)` | 方法名不同 |
| `training`参数 | `self.training`属性 | 训练模式判断方式不同 |
| `Dense(units, activation)` | `nn.Linear(in, out)` + 激活函数 | PyTorch线性层不含激活 |
| `model.build()` | 自动推断 | PyTorch无需显式build |
| `model.compile()` + `model.fit()` | 手动训练循环 | PyTorch需手动编写 |

In [ ]:
# 定义标准训练函数 / Define standard training function
def train_model(model, train_loader, valid_loader, loss_fn, optimizer,
                n_epochs=30, device=device, verbose=True):
    """
    标准PyTorch训练循环 / Standard PyTorch training loop

    Parameters:
    -----------
    model : nn.Module
        要训练的模型 / Model to train
    train_loader : DataLoader
        训练数据加载器 / Training data loader
    valid_loader : DataLoader
        验证数据加载器 / Validation data loader
    loss_fn : callable
        损失函数 / Loss function
    optimizer : optim.Optimizer
        优化器 / Optimizer
    n_epochs : int
        训练轮数 / Number of epochs
    device : torch.device
        计算设备 / Computing device
    verbose : bool
        是否打印进度 / Whether to print progress

    Returns:
    --------
    dict : 包含训练历史的字典 / Dictionary containing training history
    """
    model = model.to(device)
    history = {'train_loss': [], 'val_loss': [], 'train_mae': [], 'val_mae': []}

    for epoch in range(1, n_epochs + 1):
        # === 训练阶段 / Training phase ===
        model.train()
        train_loss_sum = 0.0
        train_mae_sum = 0.0
        train_count = 0

        for batch in train_loader:
            # 解包批次数据 / Unpack batch data
            if len(batch) == 2:
                X_batch, y_batch = batch
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                y_pred = model(X_batch)
            else:
                # 多输入情况 / Multi-input case
                X_batch = [b.to(device) for b in batch[:-1]]
                y_batch = batch[-1].to(device)
                y_pred = model(X_batch)

            loss = loss_fn(y_pred, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * y_batch.size(0)
            train_mae_sum += torch.abs(y_pred - y_batch).sum().item()
            train_count += y_batch.size(0)

        avg_train_loss = train_loss_sum / train_count
        avg_train_mae = train_mae_sum / train_count

        # === 验证阶段 / Validation phase ===
        model.eval()
        val_loss_sum = 0.0
        val_mae_sum = 0.0
        val_count = 0

        with torch.no_grad():
            for batch in valid_loader:
                if len(batch) == 2:
                    X_batch, y_batch = batch
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.to(device)
                    y_pred = model(X_batch)
                else:
                    X_batch = [b.to(device) for b in batch[:-1]]
                    y_batch = batch[-1].to(device)
                    y_pred = model(X_batch)

                loss = loss_fn(y_pred, y_batch)
                val_loss_sum += loss.item() * y_batch.size(0)
                val_mae_sum += torch.abs(y_pred - y_batch).sum().item()
                val_count += y_batch.size(0)

        avg_val_loss = val_loss_sum / val_count
        avg_val_mae = val_mae_sum / val_count

        history['train_loss'].append(avg_train_loss)
        history['train_mae'].append(avg_train_mae)
        history['val_loss'].append(avg_val_loss)
        history['val_mae'].append(avg_val_mae)

        if verbose and epoch % 5 == 0:
            print(f"Epoch {epoch:3d}/{n_epochs} | "
                  f"Train Loss: {avg_train_loss:.4f} MAE: {avg_train_mae:.4f} | "
                  f"Val Loss: {avg_val_loss:.4f} MAE: {avg_val_mae:.4f}")

    return history


def evaluate_model(model, test_loader, loss_fn, device=device):
    """
    评估模型 / Evaluate model

    Parameters:
    -----------
    model : nn.Module
        要评估的模型 / Model to evaluate
    test_loader : DataLoader
        测试数据加载器 / Test data loader
    loss_fn : callable
        损失函数 / Loss function
    device : torch.device
        计算设备 / Computing device

    Returns:
    --------
    tuple : (test_loss, test_mae)
    """
    model.eval()
    test_loss_sum = 0.0
    test_mae_sum = 0.0
    test_count = 0

    with torch.no_grad():
        for batch in test_loader:
            if len(batch) == 2:
                X_batch, y_batch = batch
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                y_pred = model(X_batch)
            else:
                X_batch = [b.to(device) for b in batch[:-1]]
                y_batch = batch[-1].to(device)
                y_pred = model(X_batch)

            loss = loss_fn(y_pred, y_batch)
            test_loss_sum += loss.item() * y_batch.size(0)
            test_mae_sum += torch.abs(y_pred - y_batch).sum().item()
            test_count += y_batch.size(0)

    avg_test_loss = test_loss_sum / test_count
    avg_test_mae = test_mae_sum / test_count
    return avg_test_loss, avg_test_mae


def count_parameters(model):
    """
    统计模型可训练参数数量 / Count trainable parameters in model

    Parameters:
    -----------
    model : nn.Module
        PyTorch模型 / PyTorch model

    Returns:
    --------
    int : 可训练参数总数 / Total number of trainable parameters
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("训练工具函数已定义 / Training utility functions defined")

## 3. SimpleRegressor — 简单自定义模型

使用`nn.Module`子类化实现一个简单的回归模型。

架构: Input → Linear(30, ReLU) → Linear(30, ReLU) → Output(1)

In [ ]:
class SimpleRegressor(nn.Module):
    """
    简单的回归模型 / Simple regression model

    架构 / Architecture: Input → Linear(30, ReLU) → Linear(30, ReLU) → Output(1)

    Keras等价 / Keras equivalent:
        class SimpleRegressor(keras.Model):
            def __init__(self, units=30, activation='relu', **kwargs):
                super().__init__(**kwargs)
                self.hidden1 = keras.layers.Dense(units, activation=activation)
                self.hidden2 = keras.layers.Dense(units, activation=activation)
                self.output_layer = keras.layers.Dense(1)
            def call(self, inputs, training=None):
                x = self.hidden1(inputs)
                x = self.hidden2(x)
                return self.output_layer(x)
    """

    def __init__(self, input_dim=8, units=30):
        """
        初始化模型层 / Initialize model layers

        Parameters:
        -----------
        input_dim : int
            输入特征维度 / Input feature dimension
        units : int
            隐藏层神经元数量 / Number of hidden units
        """
        super().__init__()
        # 注意：nn.Linear需要指定in_features和out_features
        # Note: nn.Linear requires both in_features and out_features
        self.hidden1 = nn.Linear(input_dim, units)
        self.hidden2 = nn.Linear(units, units)
        self.output_layer = nn.Linear(units, 1)

    def forward(self, x):
        """
        前向传播 / Forward pass

        Parameters:
        -----------
        x : torch.Tensor
            输入张量 / Input tensor

        Returns:
        --------
        torch.Tensor : 模型输出 / Model output
        """
        x = torch.relu(self.hidden1(x))
        x = torch.relu(self.hidden2(x))
        return self.output_layer(x)

# 创建模型实例 / Create model instance
simple_model = SimpleRegressor(input_dim=8, units=30).to(device)

# PyTorch无需显式build，直接打印模型结构 / No explicit build needed in PyTorch
print(simple_model)
print(f"\n可训练参数数量 / Trainable parameters: {count_parameters(simple_model):,}")

# 查看state_dict（等价于Keras的get_weights）/ View state_dict (equivalent to Keras get_weights)
print("\nstate_dict键 / state_dict keys:")
for key, value in simple_model.state_dict().items():
    print(f"  {key}: {value.shape}")

In [ ]:
# 训练SimpleRegressor / Train SimpleRegressor
loss_fn = nn.MSELoss()
optimizer = optim.SGD(simple_model.parameters(), lr=1e-2)

history_simple = train_model(
    simple_model, train_loader, valid_loader,
    loss_fn, optimizer, n_epochs=30
)

# 评估 / Evaluate
test_loss, test_mae = evaluate_model(simple_model, test_loader, loss_fn)
print(f"\n测试集 / Test set MSE: {test_loss:.4f}, MAE: {test_mae:.4f}")

## 4. Wide & Deep模型（子类化版本）

使用`nn.Module`子类化实现多输入多输出的Wide & Deep模型。

架构：
- Wide路径：直接连接到输出
- Deep路径：两层隐藏层 + 辅助输出
- 合并后产生主输出

In [ ]:
class WideAndDeepModel(nn.Module):
    """
    Wide & Deep模型的PyTorch实现 / PyTorch implementation of Wide & Deep model

    架构 / Architecture:
    - Wide路径 / Wide path: 直接连接到输出 / Direct connection to output
    - Deep路径 / Deep path: 两层隐藏层 + 辅助输出 / Two hidden layers + auxiliary output
    - 合并后产生主输出 / Concatenated to produce main output

    Keras等价 / Keras equivalent:
        使用 keras.layers.Concatenate() 合并，
        PyTorch中使用 torch.cat() 实现
    """

    def __init__(self, wide_dim=5, deep_dim=6, units=30):
        """
        初始化Wide & Deep模型 / Initialize Wide & Deep model

        Parameters:
        -----------
        wide_dim : int
            Wide路径输入维度 / Wide path input dimension
        deep_dim : int
            Deep路径输入维度 / Deep path input dimension
        units : int
            隐藏层神经元数量 / Number of hidden units
        """
        super().__init__()

        # Deep路径的隐藏层 / Deep path hidden layers
        self.deep_hidden1 = nn.Linear(deep_dim, units)
        self.deep_hidden2 = nn.Linear(units, units)

        # 输出层 / Output layers
        # 主输出：合并wide和deep后的维度 / Main output: dimension after concatenating wide and deep
        self.main_output = nn.Linear(wide_dim + units, 1)
        # 辅助输出：deep路径的隐藏层输出 / Auxiliary output: deep path hidden output
        self.aux_output = nn.Linear(units, 1)

    def forward(self, inputs):
        """
        前向传播 / Forward pass

        Parameters:
        -----------
        inputs : list or tuple of torch.Tensor
            [wide_input, deep_input] / Wide和Deep路径的输入

        Returns:
        --------
        tuple : (main_output, aux_output)
        """
        # 解包输入 / Unpack inputs
        input_wide, input_deep = inputs

        # Deep路径 / Deep path
        hidden1_out = torch.relu(self.deep_hidden1(input_deep))
        hidden2_out = torch.relu(self.deep_hidden2(hidden1_out))

        # 合并Wide和Deep路径 / Concatenate Wide and Deep paths
        # Keras: self.concat([input_wide, hidden2_out])
        # PyTorch: torch.cat([input_wide, hidden2_out], dim=1)
        concat_out = torch.cat([input_wide, hidden2_out], dim=1)

        # 计算输出 / Compute outputs
        main_out = self.main_output(concat_out)
        aux_out = self.aux_output(hidden2_out)

        return main_out, aux_out

# 创建模型 / Create model
wide_deep_model = WideAndDeepModel(wide_dim=5, deep_dim=6, units=30).to(device)
print(wide_deep_model)
print(f"\n可训练参数数量 / Trainable parameters: {count_parameters(wide_deep_model):,}")

In [ ]:
# 训练Wide & Deep模型 / Train Wide & Deep model
# 注意：多输出需要自定义训练循环 / Note: multi-output requires custom training loop

def train_wide_deep(model, train_loader, valid_loader, n_epochs=30,
                    main_weight=0.9, aux_weight=0.1, lr=1e-3, device=device):
    """
    训练Wide & Deep模型（多输出） / Train Wide & Deep model (multi-output)

    Parameters:
    -----------
    model : WideAndDeepModel
        Wide & Deep模型实例 / Wide & Deep model instance
    train_loader : DataLoader
        训练数据加载器（返回wide_input, deep_input, main_target, aux_target）
    valid_loader : DataLoader
        验证数据加载器
    n_epochs : int
        训练轮数 / Number of epochs
    main_weight : float
        主输出损失权重 / Main output loss weight
    aux_weight : float
        辅助输出损失权重 / Auxiliary output loss weight
    lr : float
        学习率 / Learning rate
    device : torch.device
        计算设备 / Computing device

    Returns:
    --------
    dict : 训练历史 / Training history
    """
    model = model.to(device)
    loss_fn = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)
    history = {'train_loss': [], 'val_loss': [], 'train_mae': [], 'val_mae': []}

    for epoch in range(1, n_epochs + 1):
        # 训练阶段 / Training phase
        model.train()
        train_loss_sum = 0.0
        train_mae_sum = 0.0
        train_count = 0

        for X_A, X_B, y_main, y_aux in train_loader:
            X_A, X_B = X_A.to(device), X_B.to(device)
            y_main, y_aux = y_main.to(device), y_aux.to(device)

            main_out, aux_out = model([X_A, X_B])

            # 加权损失 / Weighted loss
            # Keras: loss=['mse', 'mse'], loss_weights=[0.9, 0.1]
            loss = main_weight * loss_fn(main_out, y_main) + aux_weight * loss_fn(aux_out, y_aux)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * y_main.size(0)
            train_mae_sum += torch.abs(main_out - y_main).sum().item()
            train_count += y_main.size(0)

        avg_train_loss = train_loss_sum / train_count
        avg_train_mae = train_mae_sum / train_count

        # 验证阶段 / Validation phase
        model.eval()
        val_loss_sum = 0.0
        val_mae_sum = 0.0
        val_count = 0

        with torch.no_grad():
            for X_A, X_B, y_main, y_aux in valid_loader:
                X_A, X_B = X_A.to(device), X_B.to(device)
                y_main, y_aux = y_main.to(device), y_aux.to(device)

                main_out, aux_out = model([X_A, X_B])
                loss = main_weight * loss_fn(main_out, y_main) + aux_weight * loss_fn(aux_out, y_aux)

                val_loss_sum += loss.item() * y_main.size(0)
                val_mae_sum += torch.abs(main_out - y_main).sum().item()
                val_count += y_main.size(0)

        avg_val_loss = val_loss_sum / val_count
        avg_val_mae = val_mae_sum / val_count

        history['train_loss'].append(avg_train_loss)
        history['train_mae'].append(avg_train_mae)
        history['val_loss'].append(avg_val_loss)
        history['val_mae'].append(avg_val_mae)

        if epoch % 5 == 0:
            print(f"Epoch {epoch:3d}/{n_epochs} | "
                  f"Train Loss: {avg_train_loss:.4f} MAE: {avg_train_mae:.4f} | "
                  f"Val Loss: {avg_val_loss:.4f} MAE: {avg_val_mae:.4f}")

    return history


history_wd = train_wide_deep(wide_deep_model, train_wd_loader, valid_wd_loader, n_epochs=30)

# 评估 / Evaluate
wide_deep_model.eval()
test_loss_sum = 0.0
test_mae_sum = 0.0
test_count = 0
loss_fn = nn.MSELoss()

with torch.no_grad():
    for X_A, X_B, y_main, y_aux in test_wd_loader:
        X_A, X_B = X_A.to(device), X_B.to(device)
        y_main, y_aux = y_main.to(device), y_aux.to(device)
        main_out, aux_out = wide_deep_model([X_A, X_B])
        total_loss = 0.9 * loss_fn(main_out, y_main) + 0.1 * loss_fn(aux_out, y_aux)
        test_loss_sum += total_loss.item() * y_main.size(0)
        test_mae_sum += torch.abs(main_out - y_main).sum().item()
        test_count += y_main.size(0)

print("\n评估结果 / Evaluation results:")
print(f"总损失 / Total loss: {test_loss_sum / test_count:.4f}")
print(f"主输出MAE / Main output MAE: {test_mae_sum / test_count:.4f}")

## 5. 动态行为示例

PyTorch子类化的最大优势是可以在`forward`方法中实现动态计算逻辑，
例如条件分支、循环等Python控制流。

### training模式 vs eval模式

在PyTorch中，通过`model.train()`和`model.eval()`切换模式，
然后在`forward`中使用`self.training`属性判断当前模式。

| 操作 | Keras | PyTorch |
|------|-------|---------|
| 训练模式 | `training=True`参数 | `model.train()` + `self.training` |
| 推理模式 | `training=False`参数 | `model.eval()` + `self.training` |
| Dropout行为 | `self.dropout(x, training=training)` | `self.dropout(x)` (自动根据self.training) |

In [ ]:
class DynamicModel(nn.Module):
    """
    带有动态行为的模型示例 / Model with dynamic behavior

    特点 / Features:
    - 训练时使用Dropout / Uses Dropout during training
    - 可以根据输入动态选择计算路径 / Can dynamically select computation path

    Keras等价 / Keras equivalent:
        class DynamicModel(keras.Model):
            def __init__(self, units=30, dropout_rate=0.2, **kwargs):
                super().__init__(**kwargs)
                self.hidden1 = keras.layers.Dense(units, activation='relu')
                self.hidden2 = keras.layers.Dense(units, activation='relu')
                self.dropout = keras.layers.Dropout(dropout_rate)
                self.output_layer = keras.layers.Dense(1)
            def call(self, inputs, training=None):
                x = self.hidden1(inputs)
                x = self.dropout(x, training=training)
                x = self.hidden2(x)
                x = self.dropout(x, training=training)
                return self.output_layer(x)
    """

    def __init__(self, input_dim=8, units=30, dropout_rate=0.2):
        """
        初始化动态模型 / Initialize dynamic model

        Parameters:
        -----------
        input_dim : int
            输入特征维度 / Input feature dimension
        units : int
            隐藏层神经元数量 / Number of hidden units
        dropout_rate : float
            Dropout比率 / Dropout rate
        """
        super().__init__()
        self.hidden1 = nn.Linear(input_dim, units)
        self.hidden2 = nn.Linear(units, units)
        # nn.Dropout在self.training=True时自动激活 / nn.Dropout auto-activates when self.training=True
        self.dropout = nn.Dropout(dropout_rate)
        self.output_layer = nn.Linear(units, 1)

    def forward(self, x):
        """
        前向传播，带有动态Dropout / Forward pass with dynamic Dropout

        nn.Dropout会自动根据self.training属性决定是否激活：
        - model.train() → self.training=True → Dropout激活
        - model.eval() → self.training=False → Dropout不激活

        Keras中需要显式传递training参数，PyTorch中自动处理。

        Parameters:
        -----------
        x : torch.Tensor
            输入张量 / Input tensor

        Returns:
        --------
        torch.Tensor : 模型输出 / Model output
        """
        x = torch.relu(self.hidden1(x))
        # Dropout只在训练时生效（self.training=True时）
        # Dropout only active during training (when self.training=True)
        x = self.dropout(x)
        x = torch.relu(self.hidden2(x))
        x = self.dropout(x)
        return self.output_layer(x)

# 创建并训练动态模型 / Create and train dynamic model
dynamic_model = DynamicModel(input_dim=8, units=30, dropout_rate=0.2).to(device)
print(dynamic_model)
print(f"\n可训练参数数量 / Trainable parameters: {count_parameters(dynamic_model):,}")

# 演示training模式切换 / Demonstrate training mode switching
print("\n--- 训练模式演示 / Training mode demo ---")
dynamic_model.train()
print(f"model.train()后 self.training = {dynamic_model.training}")
dynamic_model.eval()
print(f"model.eval()后 self.training = {dynamic_model.training}")

# 训练 / Train
loss_fn = nn.MSELoss()
optimizer = optim.Adam(dynamic_model.parameters(), lr=1e-3)

history_dynamic = train_model(
    dynamic_model, train_loader, valid_loader,
    loss_fn, optimizer, n_epochs=30
)

# 评估 / Evaluate
test_loss, test_mae = evaluate_model(dynamic_model, test_loader, loss_fn)
print(f"\n测试集 / Test set MSE: {test_loss:.4f}, MAE: {test_mae:.4f}")

## 6. 自定义层

除了自定义模型，还可以创建自定义层，实现更细粒度的控制。

在PyTorch中，自定义层同样继承`nn.Module`（而非Keras中的`keras.layers.Layer`）。

In [ ]:
class ResidualBlock(nn.Module):
    """
    残差块（Residual Block）自定义层 / Residual Block custom layer

    实现 / Implementation: output = activation(input + Linear(Linear(input)))

    Keras等价 / Keras equivalent:
        class ResidualBlock(keras.layers.Layer):
            def __init__(self, units, activation='relu', **kwargs):
                super().__init__(**kwargs)
                self.units = units
                self.activation = keras.activations.get(activation)
            def build(self, input_shape):
                self.dense1 = keras.layers.Dense(self.units, activation='relu')
                self.dense2 = keras.layers.Dense(input_shape[-1])
                super().build(input_shape)
            def call(self, inputs):
                x = self.dense1(inputs)
                x = self.dense2(x)
                return self.activation(inputs + x)

    注意 / Note:
    Keras中自定义层继承keras.layers.Layer，PyTorch中统一继承nn.Module。
    Keras需要实现build()方法延迟创建权重，PyTorch在__init__中直接创建。
    """

    def __init__(self, input_dim, units=30):
        """
        初始化残差块 / Initialize residual block

        Parameters:
        -----------
        input_dim : int
            输入特征维度（输出维度与之相同以实现残差连接）
            Input feature dimension (output dimension matches for residual connection)
        units : int
            中间隐藏层神经元数量 / Number of intermediate hidden units
        """
        super().__init__()
        # PyTorch在__init__中直接创建所有层，无需延迟build
        # PyTorch creates all layers in __init__, no deferred build needed
        self.dense1 = nn.Linear(input_dim, units)
        self.dense2 = nn.Linear(units, input_dim)  # 输出维度与输入相同 / Output dim matches input

    def forward(self, x):
        """
        前向传播：残差连接 / Forward pass: residual connection

        Parameters:
        -----------
        x : torch.Tensor
            输入张量 / Input tensor

        Returns:
        --------
        torch.Tensor : 残差连接后的输出 / Output after residual connection
        """
        residual = x
        x = torch.relu(self.dense1(x))
        x = self.dense2(x)
        return torch.relu(residual + x)  # 残差连接 / Residual connection

# 测试残差块 / Test residual block
test_block = ResidualBlock(input_dim=8, units=30).to(device)
test_input = torch.randn(4, 8).to(device)
test_output = test_block(test_input)
print("残差块测试 / Residual block test:")
print(f"输入形状 / Input shape: {test_input.shape}")
print(f"输出形状 / Output shape: {test_output.shape}")
print(f"可训练参数 / Trainable params: {count_parameters(test_block):,}")

In [ ]:
class ResNet(nn.Module):
    """
    使用残差块的模型 / Model using residual blocks

    架构 / Architecture:
    Input → Linear(units, ReLU) → [ResidualBlock × n_blocks] → Linear(1)

    Keras等价 / Keras equivalent:
        class ResNet(keras.Model):
            def __init__(self, n_blocks=3, units=30, **kwargs):
                super().__init__(**kwargs)
                self.input_dense = keras.layers.Dense(units, activation='relu')
                self.res_blocks = [ResidualBlock(units) for _ in range(n_blocks)]
                self.output_layer = keras.layers.Dense(1)
            def call(self, inputs, training=None):
                x = self.input_dense(inputs)
                for block in self.res_blocks:
                    x = block(x)
                return self.output_layer(x)

    注意 / Note:
    PyTorch中使用nn.ModuleList注册子模块列表，确保参数被正确追踪。
    如果使用普通Python列表，参数不会被optimizer识别。
    """

    def __init__(self, input_dim=8, n_blocks=3, units=30):
        """
        初始化ResNet模型 / Initialize ResNet model

        Parameters:
        -----------
        input_dim : int
            输入特征维度 / Input feature dimension
        n_blocks : int
            残差块数量 / Number of residual blocks
        units : int
            隐藏层神经元数量 / Number of hidden units
        """
        super().__init__()
        self.input_dense = nn.Linear(input_dim, units)

        # 重要：使用nn.ModuleList而非普通Python列表！
        # Important: Use nn.ModuleList, not a plain Python list!
        # 普通列表中的参数不会被model.parameters()追踪
        # Parameters in a plain list won't be tracked by model.parameters()
        self.res_blocks = nn.ModuleList([
            ResidualBlock(input_dim=units, units=units) for _ in range(n_blocks)
        ])

        self.output_layer = nn.Linear(units, 1)

    def forward(self, x):
        """
        前向传播 / Forward pass

        Parameters:
        -----------
        x : torch.Tensor
            输入张量 / Input tensor

        Returns:
        --------
        torch.Tensor : 模型输出 / Model output
        """
        x = torch.relu(self.input_dense(x))
        for block in self.res_blocks:
            x = block(x)
        return self.output_layer(x)

# 创建并训练ResNet / Create and train ResNet
resnet = ResNet(input_dim=8, n_blocks=3, units=30).to(device)
print(resnet)
print(f"\n可训练参数数量 / Trainable parameters: {count_parameters(resnet):,}")

# 对比ModuleList vs 普通列表 / Compare ModuleList vs plain list
print("\n--- nn.ModuleList的重要性 / Importance of nn.ModuleList ---")
print(f"ResNet参数数量（使用ModuleList）: {count_parameters(resnet):,}")

# 错误示范：使用普通列表 / Wrong example: using plain list
class BadResNet(nn.Module):
    def __init__(self, input_dim=8, n_blocks=3, units=30):
        super().__init__()
        self.input_dense = nn.Linear(input_dim, units)
        self.res_blocks = [  # 普通Python列表，参数不会被追踪！
            ResidualBlock(input_dim=units, units=units) for _ in range(n_blocks)
        ]
        self.output_layer = nn.Linear(units, 1)

    def forward(self, x):
        x = torch.relu(self.input_dense(x))
        for block in self.res_blocks:
            x = block(x)
        return self.output_layer(x)

bad_resnet = BadResNet(input_dim=8, n_blocks=3, units=30)
print(f"BadResNet参数数量（使用普通列表）: {count_parameters(bad_resnet):,}")
print("→ 残差块的参数丢失了！/ Residual block parameters are lost!")

# 训练ResNet / Train ResNet
loss_fn = nn.MSELoss()
optimizer = optim.Adam(resnet.parameters(), lr=1e-3)

history_resnet = train_model(
    resnet, train_loader, valid_loader,
    loss_fn, optimizer, n_epochs=30
)

test_loss, test_mae = evaluate_model(resnet, test_loader, loss_fn)
print(f"\n测试集 / Test set MSE: {test_loss:.4f}, MAE: {test_mae:.4f}")

## 7. 可视化训练过程

In [ ]:
# 对比不同模型的训练曲线 / Compare training curves of different models
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 损失曲线 / Loss curves
axes[0].plot(history_simple['train_loss'], label='Simple')
axes[0].plot(history_dynamic['train_loss'], label='Dynamic (Dropout)')
axes[0].plot(history_resnet['train_loss'], label='ResNet')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 验证损失 / Validation loss
axes[1].plot(history_simple['val_loss'], label='Simple')
axes[1].plot(history_dynamic['val_loss'], label='Dynamic (Dropout)')
axes[1].plot(history_resnet['val_loss'], label='ResNet')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Loss')
axes[1].set_title('Validation Loss Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. TF vs PyTorch 对照

### 模型定义对照

| 概念 | TensorFlow/Keras | PyTorch |
|------|-----------------|---------|
| 模型基类 | `keras.Model` | `nn.Module` |
| 自定义层基类 | `keras.layers.Layer` | `nn.Module` |
| 前向传播方法 | `call(self, inputs, training=None)` | `forward(self, x)` |
| 训练模式判断 | `training`参数 | `self.training`属性 |
| 全连接层 | `Dense(units, activation)` | `nn.Linear(in, out)` + 激活函数 |
| 拼接层 | `Concatenate()` | `torch.cat(tensors, dim)` |
| Dropout | `Dropout(rate)` | `nn.Dropout(p)` |
| 激活函数 | 字符串或`Activation`层 | `torch.relu`等函数式API |

### 训练流程对照

| 概念 | TensorFlow/Keras | PyTorch |
|------|-----------------|---------|
| 编译模型 | `model.compile(loss, optimizer, metrics)` | 手动定义`loss_fn`和`optimizer` |
| 训练 | `model.fit(X, y, epochs)` | 手动训练循环 |
| 评估 | `model.evaluate(X, y)` | 手动评估循环 |
| 预测 | `model.predict(X)` | `model(X)` |
| 模式切换 | 自动 | `model.train()` / `model.eval()` |
| 梯度计算 | 自动 | `loss.backward()` + `optimizer.step()` |
| 梯度清零 | 自动 | `optimizer.zero_grad()` |

### 模型保存与序列化对照

| 概念 | TensorFlow/Keras | PyTorch |
|------|-----------------|---------|
| 获取权重 | `model.get_weights()` | `model.state_dict()` |
| 设置权重 | `model.set_weights(w)` | `model.load_state_dict(sd)` |
| 保存模型 | `model.save('path')` | `torch.save(model.state_dict(), 'path')` |
| 加载模型 | `keras.models.load_model('path')` | `model.load_state_dict(torch.load('path'))` |
| 模型配置 | `get_config()` | 无内置方法，需自定义 |

### 子模块注册对照

| 概念 | TensorFlow/Keras | PyTorch |
|------|-----------------|---------|
| 属性赋值 | 自动追踪 | 自动追踪 |
| 列表存储 | 普通Python列表即可 | 必须使用`nn.ModuleList` |
| 字典存储 | 普通Python字典即可 | 必须使用`nn.ModuleDict` |
| 延迟构建 | `build()`方法 | 在`__init__`中直接指定维度 |

### 设计哲学差异

1. **Keras**："配置优先"——通过`compile`/`fit`封装训练细节，子类化API仍保留高层接口
2. **PyTorch**："显式优先"——所有训练步骤都需要手动编写，但控制力更强
3. **Keras**的`Dense`包含激活函数，**PyTorch**的`Linear`不包含，需手动应用
4. **Keras**通过`training`参数传递模式，**PyTorch**通过`self.training`属性自动管理
5. **Keras**子类化需要`build()`延迟创建权重，**PyTorch**在`__init__`中直接指定维度

## 9. 注意事项与最佳实践

### nn.Module子类化的优势

1. **完全的灵活性**: 可以实现任意复杂的前向传播逻辑
2. **动态计算图**: 支持条件分支、循环等控制流（PyTorch原生动态图）
3. **易于调试**: 可以在forward方法中使用print或断点，与普通Python代码无异
4. **研究友好**: 适合实现新的模型架构

### nn.Module子类化的限制

1. **需要手动编写训练循环**: 没有Keras的`compile`/`fit`便利
2. **需要显式指定维度**: `nn.Linear`需要`in_features`和`out_features`
3. **子模块注册需注意**: 列表用`nn.ModuleList`，字典用`nn.ModuleDict`
4. **序列化需额外处理**: `state_dict`只保存权重，不保存模型结构

### 选择建议

- **日常使用**: 优先使用`nn.Sequential`构建简单模型
- **复杂架构**: 使用`nn.Module`子类化
- **研究实验**: `nn.Module`子类化最适合快速原型开发
- **生产部署**: 考虑使用`torch.jit.script`或`torch.jit.trace`优化

## 10. 练习

### 练习1：实现带Skip Connection的MLP

创建一个`SkipMLP`模型，每隔2层添加一次跳跃连接（类似DenseNet的思想）：
- 输入 → Linear(64, ReLU) → Linear(64, ReLU) → **与输入拼接** → Linear(64, ReLU) → Linear(1)
- 提示：使用`torch.cat`实现拼接，注意维度变化

```python
class SkipMLP(nn.Module):
    def __init__(self, input_dim=8, hidden_dim=64):
        super().__init__()
        # TODO: 定义各层
        pass
    
    def forward(self, x):
        # TODO: 实现前向传播，包含跳跃连接
        pass
```

### 练习2：实现条件计算模型

创建一个`ConditionalModel`，根据输入的某个特征值选择不同的计算路径：
- 如果`x[:, 0] > 0`，使用路径A（2层窄网络）
- 否则使用路径B（1层宽网络）
- 提示：PyTorch的动态图天然支持条件分支，直接使用`if/else`即可

```python
class ConditionalModel(nn.Module):
    def __init__(self, input_dim=8):
        super().__init__()
        # TODO: 定义路径A和路径B的层
        pass
    
    def forward(self, x):
        # TODO: 根据条件选择路径
        # 注意：条件分支在PyTorch中是合法的，这是动态图的优势
        pass
```

### 练习3：实现自定义BatchNorm层

不使用`nn.BatchNorm1d`，手动实现一个BatchNorm层：
- 训练时：使用当前batch的均值和方差，并更新运行均值和方差
- 推理时：使用运行均值和方差
- 提示：使用`self.register_buffer`注册运行统计量

```python
class CustomBatchNorm(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        # TODO: 定义可学习参数gamma、beta和运行统计量
        pass
    
    def forward(self, x):
        # TODO: 根据self.training选择不同的计算方式
        pass
```

## 小结

### 核心要点

1. **继承`nn.Module`**: PyTorch中模型和自定义层都继承`nn.Module`（Keras区分`Model`和`Layer`）
2. **`__init__`定义组件**: 所有可训练层都应在初始化时创建，列表用`nn.ModuleList`
3. **`forward`实现逻辑**: 前向传播的完整计算流程，无需`training`参数
4. **`self.training`属性**: 通过`model.train()`/`model.eval()`切换，`nn.Dropout`等自动响应
5. **手动训练循环**: PyTorch需要显式编写`zero_grad`→`backward`→`step`流程
6. **`state_dict`管理权重**: 等价于Keras的`get_weights`/`set_weights`

### 何时使用nn.Module子类化

- 需要动态计算图（如条件计算、循环展开）
- 实现复杂的注意力机制
- 研究新的网络架构
- 需要在前向传播中使用Python控制流
- 需要多输入多输出的复杂拓扑结构